<a href="https://colab.research.google.com/github/DiasFrazerGroup/Intro-to-probabilistic-modelling-2026/blob/main/Session2b_ProteinVAE_WithAnswers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A Protein Variational Autoencoder — From Scratch
*Session 2b — An Introduction to Probabilistic Model Building*

---

## The basic idea behind EVE and other unsupervised models for protein variant effect prediction

> If you train a VAE on a multiple sequence alignment (MSA) of a protein family,  
> the model learns the **evolutionary grammar** of that protein —  
> which amino acids are tolerated at each position, and which co-evolve together.  
> A variant that violates this grammar (low ELBO) is likely **functionally damaging**.  
> A variant that respects it (high ELBO) is likely **benign**.

The ELBO from the VAE is what we use as a variant scoring function  
$$\text{EVE score}(\text{mut}) \propto \underbrace{\text{ELBO}(\text{mut})}_{\text{mutant}} - \underbrace{\text{ELBO}(\text{wt})}_{\text{wildtype}}$$

No 3D structure. No biochemistry lab. Just sequences + a VAE + Bayesian reasoning.

---

## What we'll build

Rather than downloading a pre-trained EVE model, we'll **build a simplified version scratch** —  
which means you'll understand every component.

1. **Synthetic MSA**: protein sequences generated from a known "evolutionary grammar" with co-evolving positions and functional constraints
2. **Protein VAE**: a small VAE that treats each sequence as a data point and learns the grammar
3. **Latent space**: visualise how the VAE organises sequences in 2D
4. **Variant scoring**: score mutations by their ELBO, compare to known ground truth

Because we generate the data ourselves, we **know the ground truth** — which positions matter and which don't.  
This lets us verify that the VAE has correctly learned the evolutionary grammar.

**Session 2a → 2b bridge:** In Session 2a, $p(\mathbf{x}|z)$ was Normal with known $\sigma$.
Here $p(\mathbf{x}|z)$ is a deep neural network — the decoder learns the likelihood from data.
The ELBO objective and the DAG structure are unchanged.

In [ ]:
!pip install torch matplotlib numpy plotly --quiet

In [ ]:
import torch                          # PyTorch: the main deep learning library we use throughout
import torch.nn as nn                  # nn = "neural networks" — building blocks like Linear layers, ReLU, etc.
import torch.nn.functional as F        # F = standalone functions (cross_entropy, relu) used inside the model
from torch.utils.data import DataLoader, TensorDataset
# DataLoader: feeds data to the model in small batches during training (avoids loading all at once)
# TensorDataset: wraps two arrays (inputs + labels) so they stay paired when shuffled
import numpy as np                     # NumPy: fast numerical arrays — most pre-processing happens here
import matplotlib.pyplot as plt        # Standard plotting library
import matplotlib.cm as cm
from umap import UMAP                  # UMAP: reduces high-dimensional latent spaces to 2D for visualisation

# Detect whether a GPU is available.
# 'cuda' = NVIDIA GPU (10-100× faster for training neural networks).
# 'cpu'  = falls back to the processor — always works, just slower.
# On Colab: Runtime → Change runtime type → T4 GPU to get 'cuda'.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

## 1. Generating a synthetic Multiple Sequence Alignment

Real protein MSAs contain thousands of homologous sequences from different organisms.  
Here we'll generate a synthetic MSA with a **known grammar** so we can verify that the VAE learns it.

### Our evolutionary grammar

We'll model a protein of length **L = 30** with a vocabulary of **7 amino acids** (simplified).

The grammar has three types of constraints:

| Rule | Positions | Description |
|---|---|---|
| **Co-evolution** | (4, 14) and (9, 19) | These pairs must match: if position 4 = A, then position 14 must also = A (epistatic coupling) |
| **Functional site** | 7 | Only 2 amino acids allowed at this position (strong conservation) |
| **Free positions** | all others | All amino acids equally likely |

The co-evolving pairs simulate real proteins where residues in contact must be co-compatible.  
The single functional site means each of the 49 co-evolving clusters splits into exactly **2 sub-groups** (blue vs orange) — clean enough to see within each cluster in the latent space.

In [ ]:
rng = np.random.default_rng(42)   # Fixed random seed → same sequences every run (reproducibility)

# --- Grammar parameters ---
L = 30          # Protein length: 30 amino acid positions (real proteins are typically 100–1000)
V = 7           # Vocabulary size: we use 7 simplified amino acid types (labelled 0–6)
N_train = 8000  # Number of sequences in our synthetic MSA

# Co-evolving pairs: positions that must always carry the SAME amino acid.
# This mimics real proteins where two residues are in direct contact —
# if one changes, the other must change to maintain the interaction.
coevolving_pairs = [(4, 14), (9, 19)]

# Functional site: position that can only carry amino acids 0 or 1.
# This mimics highly conserved active-site residues that are intolerant to substitution.
functional_sites = [7]
allowed_at_functional = {7: [0, 1]}   # Only these 2 amino acids are allowed at position 7

def generate_sequence(rng):
    """Generate one protein sequence that obeys the evolutionary grammar."""
    # Step 1: start with a completely random sequence (each position drawn uniformly from 0-6)
    seq = rng.integers(0, V, size=L)

    # Step 2: enforce co-evolution — copy the amino acid from position i to position j
    for i, j in coevolving_pairs:
        seq[j] = seq[i]   # j is "enslaved" to i — they must match

    # Step 3: enforce functional site conservation — overwrite with an allowed amino acid
    for pos in functional_sites:
        seq[pos] = rng.choice(allowed_at_functional[pos])   # picks 0 or 1 with equal probability

    return seq

# Generate the full MSA: call generate_sequence() N_train times and stack into a 2D array
# Shape: (N_train, L) = (8000 sequences, 30 positions)
msa = np.array([generate_sequence(rng) for _ in range(N_train)])
print(f"MSA shape: {msa.shape}  ({N_train} sequences × {L} positions)")
print(f"Vocabulary: {V} amino acids (0-{V-1})")

# Sanity checks — verify the grammar constraints are satisfied in the generated data
for i, j in coevolving_pairs:
    match_rate = (msa[:, i] == msa[:, j]).mean()   # fraction of sequences where positions i and j agree
    print(f"  Co-evolution ({i},{j}): match rate = {match_rate:.2f} (should be 1.0)")
for pos in functional_sites:
    unique = np.unique(msa[:, pos])   # what amino acids actually appear at this position?
    print(f"  Functional site {pos}: unique AAs = {unique} (should be [0,1])")

In [ ]:
import matplotlib.colors as mcolors

fig, ax = plt.subplots(figsize=(10, 6))

cmap_disc = plt.get_cmap('tab10', V)
norm_disc  = mcolors.BoundaryNorm(np.arange(-0.5, V, 1), V)

# rows = sequences, columns = positions (standard MSA orientation)
im = ax.imshow(msa[:80], aspect='auto',
               cmap=cmap_disc, norm=norm_disc,
               interpolation='nearest')

ax.set_xlabel('Position')
ax.set_ylabel('Sequence index')
ax.set_title('First 80 sequences of the MSA (columns = positions, colour = amino acid)')

# Label every position on the x-axis
ax.set_xticks(range(L))
ax.set_xticklabels(range(L), fontsize=7)

# Colour special position labels
special_colors = {}
for i, j in coevolving_pairs:
    special_colors[i] = 'red'
    special_colors[j] = 'red'
for pos in functional_sites:
    special_colors[pos] = 'orange'

for label in ax.get_xticklabels():
    pos = int(label.get_text())
    if pos in special_colors:
        label.set_color(special_colors[pos])
        label.set_fontweight('bold')

# Discrete colorbar
cbar = plt.colorbar(im, ax=ax, ticks=range(V))
cbar.set_label('Amino acid')
cbar.set_ticklabels([f'AA {i}' for i in range(V)])

from matplotlib.patches import Patch
handles = [
    Patch(facecolor='none', edgecolor='red',    label='Co-evolving positions (red labels)'),
    Patch(facecolor='none', edgecolor='orange', label='Functional sites (orange labels)'),
]
ax.legend(handles=handles, loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Pairwise mutual information — shows which positions co-vary in the MSA
def pairwise_mi(msa, V):
    L  = msa.shape[1]
    N  = msa.shape[0]
    mi = np.zeros((L, L))
    for i in range(L):
        for j in range(i + 1, L):
            joint = np.zeros((V, V))
            np.add.at(joint, (msa[:, i], msa[:, j]), 1)
            joint /= N
            pi   = joint.sum(axis=1, keepdims=True)
            pj   = joint.sum(axis=0, keepdims=True)
            mask = joint > 0
            mi[i, j] = (joint[mask] * np.log(joint[mask] / (pi * pj)[mask])).sum()
            mi[j, i] = mi[i, j]
    return mi

mi_matrix = pairwise_mi(msa, V)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(mi_matrix, cmap='hot_r', interpolation='nearest')
plt.colorbar(im, ax=ax, label='Mutual information (bits)')

# Highlight the co-evolving pairs
for i, j in coevolving_pairs:
    ax.add_patch(plt.Rectangle((j-0.5, i-0.5), 1, 1, fill=False, edgecolor='cyan', lw=2))
    ax.add_patch(plt.Rectangle((i-0.5, j-0.5), 1, 1, fill=False, edgecolor='cyan', lw=2))

ax.set_xlabel('Position')
ax.set_ylabel('Position')
ax.set_title('Pairwise mutual information in the MSA\n(cyan boxes = co-evolving pairs)')
plt.tight_layout()
plt.show()

print("Co-evolving pairs should have the highest off-diagonal MI.")
print("Free positions should have near-zero MI with everything.")

## 2. The Protein VAE

### Architecture

Our VAE treats each protein sequence as a data point.  
The model consists of:

```
Input x: one-hot encoded sequence  (L × V)  →  flattened to (L*V,)

Encoder q(z|x):  x → [FC → ReLU → FC → ReLU] → (μ, log σ²)
                                                       ↓
                                              z ~ N(μ, σ²)   (reparameterisation)

Decoder p(x|z):  z → [FC → ReLU → FC] → logits  →  Categorical over amino acids
```

### Training objective: ELBO

$$\mathcal{L}(\theta, \phi; x) = \underbrace{\mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x|z)]}_{\text{reconstruction}} - \underbrace{D_{\text{KL}}(q_\phi(z|x) \| p(z))}_{\text{KL regularisation}}$$

- **Reconstruction**: does the decoder recover the input sequence?
- **KL**: does the posterior $q(z|x)$ stay close to the prior $\mathcal{N}(0,I)$?

This is exactly the ELBO from the lecture — maximising it is equivalent to maximising a lower bound on $\log p(x)$.

In [ ]:
# In Python, a "class" is a blueprint for creating objects that bundle data + functions together.
# nn.Module is PyTorch's base class for all neural networks.
# By writing "class ProteinVAE(nn.Module):" we inherit all of PyTorch's training machinery.
class ProteinVAE(nn.Module):

    def __init__(self, seq_len, vocab_size, latent_dim=2, hidden_dim=256):
        """Set up the network architecture (called once when you create the model)."""
        super().__init__()   # Required: initialises the parent nn.Module class
        self.seq_len    = seq_len
        self.vocab_size = vocab_size
        self.latent_dim = latent_dim
        input_dim = seq_len * vocab_size   # one-hot flattened: L*V = 30*7 = 210

        # --- ENCODER: sequence → (μ, log σ²) ---
        # nn.Sequential chains layers: input flows through each in order.
        # This is the "recognition network" q(z|x) — it compresses the sequence into a small code.
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            # nn.Linear: a fully-connected layer.
            # Each of the hidden_dim output neurons computes: w·x + b
            # (a weighted sum of all inputs, plus a bias term).
            nn.ReLU(),
            # ReLU (Rectified Linear Unit): activation function — replaces negatives with 0.
            # ReLU(x) = max(0, x).
            # Without activation functions, stacking Linear layers is still just one linear operation.
            # ReLU introduces the non-linearity that lets deep networks learn complex patterns.
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        # Two separate output heads: one for the mean, one for the log-variance of q(z|x)
        self.fc_mu      = nn.Linear(hidden_dim, latent_dim)   # μ: the centre of the posterior
        self.fc_log_var = nn.Linear(hidden_dim, latent_dim)   # log σ²: the spread (log for numerical stability)

        # --- DECODER: z → amino acid logits ---
        # This is the "generative network" p(x|z) — it reconstructs the sequence from the code.
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            # Output: input_dim = L*V values, which we reshape to (L, V) logits.
            # Logit = raw score before converting to a probability.
            # For each position we get V logits; the softmax over them gives P(aa | z, position).
        )

    def encode(self, x):
        """Map a batch of one-hot sequences to (μ, log σ²) of the posterior q(z|x).
        x: (batch, L*V)  →  returns μ: (batch, latent_dim), log_var: (batch, latent_dim)
        """
        h = self.encoder(x)                       # (batch, hidden_dim)
        return self.fc_mu(h), self.fc_log_var(h)  # two separate linear projections

    def reparameterise(self, mu, log_var):
        """Sample z ~ q(z|x) = N(μ, σ²) using the reparameterisation trick.

        Why not just sample directly from N(μ, σ²)?
        Because gradients can't flow through a random sample.
        The trick: write z = μ + σ·ε  where ε ~ N(0,1).
        The randomness is in ε (which we don't differentiate through),
        so gradients flow through μ and σ unobstructed.
        """
        std = torch.exp(0.5 * log_var)    # σ = exp(log σ² / 2)  — always positive
        eps = torch.randn_like(std)        # ε ~ N(0,1), same shape as std
        return mu + eps * std              # z ~ N(μ, σ²)

    def decode(self, z):
        """Map latent code z to amino acid logits.
        z: (batch, latent_dim)  →  logits: (batch, seq_len, vocab_size)
        """
        logits_flat = self.decoder(z)                           # (batch, L*V)
        return logits_flat.view(-1, self.seq_len, self.vocab_size)  # reshape to (batch, L, V)

    def forward(self, x):
        """Full forward pass: encode → reparameterise → decode.
        Returns everything needed for the training loss.
        """
        mu, log_var = self.encode(x)
        z           = self.reparameterise(mu, log_var)
        logits      = self.decode(z)
        return logits, mu, log_var, z

    def elbo(self, x_onehot_flat, x_indices, n_samples=10):
        """Compute the ELBO for each sequence in the batch (used for variant scoring, not training).

        ELBO = E[log p(x|z)] - KL(q(z|x) || p(z))
             = reconstruction term - KL term

        x_onehot_flat: (batch, L*V)  — one-hot encoded input
        x_indices:     (batch, L)    — integer amino acid indices (for cross-entropy)
        n_samples:     how many z samples to average over (more = more accurate but slower)
        Returns: per-sequence ELBO, shape (batch,)
        """
        mu, log_var = self.encode(x_onehot_flat)

        # --- Reconstruction term: E_{z~q}[log p(x|z)] ---
        # Estimated by Monte Carlo: sample z n_samples times and average log p(x|z)
        recon_sum = torch.zeros(x_onehot_flat.shape[0], device=x_onehot_flat.device)
        for _ in range(n_samples):
            z      = self.reparameterise(mu, log_var)   # draw one z sample
            logits = self.decode(z)                      # (batch, L, V)

            # cross_entropy computes -log P(correct aa | logits) at each position
            # We negate it to get +log p(x|z), then sum over positions
            log_px_z = -F.cross_entropy(
                logits.view(-1, self.vocab_size),   # reshape to (batch*L, V) for cross_entropy
                x_indices.view(-1),                  # reshape to (batch*L,)
                reduction='none'                     # keep per-position values (don't average)
            ).view(x_onehot_flat.shape[0], self.seq_len).sum(dim=1)   # sum over L positions → (batch,)
            recon_sum += log_px_z
        recon = recon_sum / n_samples   # average over z samples

        # --- KL term: KL(q(z|x) || p(z)) where p(z) = N(0,I) ---
        # Closed-form formula for KL between two Gaussians:
        # KL = -0.5 * sum(1 + log σ² - μ² - σ²)
        kl = -0.5 * (1 + log_var - mu.pow(2) - log_var.exp()).sum(dim=1)   # (batch,)

        return recon - kl   # ELBO = reconstruction - KL  (higher is better)


def compute_elbo_loss(logits, x_indices, mu, log_var):
    """Training loss = negative ELBO, averaged over the batch.

    We minimise this during training (PyTorch minimises by convention),
    which is equivalent to maximising the ELBO.
    """
    # Reconstruction loss: cross-entropy summed over all L positions, then averaged over batch
    # cross_entropy = -log P(correct amino acid) = how surprised the model is by the true sequence
    recon = F.cross_entropy(
        logits.view(-1, logits.shape[-1]),   # (batch*L, V)
        x_indices.view(-1),                   # (batch*L,)
        reduction='none'
    ).view(logits.shape[0], logits.shape[1]).sum(dim=1).mean()   # sum over positions, mean over batch

    # KL divergence: same formula as above, averaged over batch
    kl = -0.5 * (1 + log_var - mu.pow(2) - log_var.exp()).sum(dim=1).mean()

    return recon + kl   # negative ELBO (we minimise this)

print("VAE class defined.")

## 3. Preparing data and training

We one-hot encode each sequence and train with Adam.  
We use **latent_dim = 2** so we can directly visualise the latent space without UMAP —  
just plot $z_1$ vs $z_2$ and colour by properties of the sequence.

In [ ]:
def one_hot(seq_batch, vocab_size):
    """Convert integer-coded sequences to one-hot encoding.

    Why one-hot? Neural networks work with continuous numbers, not categories.
    One-hot represents amino acid 3 (out of 7) as [0, 0, 0, 1, 0, 0, 0] —
    a vector with a 1 at position 3 and 0s everywhere else.
    This avoids implying any ordering between amino acid types.

    seq_batch: (N, L) integer array  →  returns (N, L*V) float array
    """
    N, L = seq_batch.shape
    oh = np.zeros((N, L, vocab_size), dtype=np.float32)   # start with all zeros: (N, L, V)
    for i in range(N):          # loop over sequences
        for j in range(L):      # loop over positions
            oh[i, j, seq_batch[i, j]] = 1.0   # set the correct amino acid slot to 1
    return oh.reshape(N, -1)    # flatten (L, V) → (L*V,) for each sequence: (N, L*V)

# --- Encode the training MSA ---
X_oh  = one_hot(msa, V)                       # numpy array: (N_train, L*V) = (8000, 210)
X_idx = torch.tensor(msa, dtype=torch.long)   # PyTorch tensor of integer labels: (N_train, L)
X_oh_t = torch.tensor(X_oh)                   # PyTorch tensor of one-hot encodings: (N_train, L*V)

# TensorDataset pairs the one-hot input with the integer labels — they stay in sync during shuffling
dataset = TensorDataset(X_oh_t, X_idx)

# DataLoader wraps the dataset and handles:
#   - splitting into batches of 256 sequences
#   - shuffling the order each epoch (shuffle=True)
# During training, we iterate over the loader one batch at a time
loader = DataLoader(dataset, batch_size=256, shuffle=True)

print(f"One-hot tensor shape: {X_oh_t.shape}")
# Should be (8000, 210) = (N_train, L*V)

In [ ]:
# Create the model and move it to the GPU (if available)
model     = ProteinVAE(seq_len=L, vocab_size=V, latent_dim=2, hidden_dim=256).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# ReduceLROnPlateau halves the learning rate when loss stops improving for 10 epochs.
# Helps the model converge cleanly even if the initial lr was slightly too large.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=10
)

n_epochs = 120
losses   = []

for epoch in range(1, n_epochs + 1):

    model.train()
    epoch_loss = 0.0

    for x_oh, x_idx in loader:
        x_oh  = x_oh.to(device)
        x_idx = x_idx.to(device)

        logits, mu, log_var, z = model(x_oh)
        loss = compute_elbo_loss(logits, x_idx, mu, log_var)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * x_oh.shape[0]

    epoch_loss /= N_train
    losses.append(epoch_loss)
    scheduler.step(epoch_loss)   # reduce lr if loss has plateaued

    if epoch % 20 == 0:
        lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch:3d} | loss: {epoch_loss:.3f} | lr: {lr:.2e}")

plt.figure(figsize=(7, 3))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Negative ELBO (lower = better)')
plt.title('Training loss — should decrease and level off')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Exploring the latent space

Since `latent_dim = 2`, we can directly plot $z_1$ vs $z_2$ for all training sequences.  
We'll colour the points by properties of each sequence:

1. **Amino acid at position 7** (functional site — should only be 0 or 1)
2. **Amino acid at position 4** (one of the co-evolving pair)

If the VAE has learned the grammar, sequences with different amino acids at key positions  
should be **separated in the latent space**.

In [ ]:
model.eval()   # Switch to "evaluation mode" — disables dropout and batch-norm updates (not critical here,
               # but always good practice when you're not training)

# Encode all training sequences to get their latent coordinates
# torch.no_grad() tells PyTorch not to track gradients — saves memory and speeds things up
# (we don't need gradients when just running inference)
with torch.no_grad():
    mu_all, _ = model.encode(X_oh_t.to(device))   # mu_all: (N_train, 2) — posterior means
    z_all = mu_all.cpu().numpy()                   # move to CPU and convert to numpy for plotting

# Subsample 2000 points to keep the scatter plot readable
idx_plot = rng.choice(N_train, size=2000, replace=False)
z_plot   = z_all[idx_plot]       # (2000, 2) — latent coordinates
msa_plot = msa[idx_plot]         # (2000, L) — corresponding sequences

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Each panel colours the same 2D latent space by a different property of the sequences.
# This lets us ask: "does the VAE separate sequences based on this constraint?"
panels = [
    (7,  'Functional site (pos 7)\n— only 2 AAs allowed'),
    (4,  'Co-evolving pos 4\n— encodes (4,14) pair along one axis'),
    (9,  'Co-evolving pos 9\n— encodes (9,19) pair along orthogonal axis'),
    (0,  'Free position (pos 0)\n— no constraint, no structure'),
]

for ax, (pos, title) in zip(axes, panels):
    c  = msa_plot[:, pos]    # amino acid at this position for each plotted sequence: (2000,)
    sc = ax.scatter(
        z_plot[:, 0], z_plot[:, 1],   # z₁ vs z₂
        c=c, cmap='tab10',             # colour by amino acid type (tab10 has 10 distinct colours)
        vmin=0, vmax=V-1,              # fix colour scale so AA 0 is always the same colour across panels
        alpha=0.5, s=8                 # semi-transparent small dots (many overlapping points)
    )
    plt.colorbar(sc, ax=ax, label=f'AA at pos {pos}')
    ax.set_title(title, fontsize=9)
    ax.set_xlabel('z₁'); ax.set_ylabel('z₂')
    ax.grid(alpha=0.2)

plt.suptitle(
    'Latent space structure\n'
    'Pos 4 and pos 9 cluster along orthogonal axes — the VAE disentangles the two co-evolving pairs',
    fontsize=11)
plt.tight_layout()
plt.show()

### What to look for

**Panel 1 — Functional site (pos 7):** Only AA 0 and 1 appear. Look for a faint blue/orange split *inside* each of the 49 co-evolving clusters. It won't be as sharp as the co-evolving structure — the 2D space is already occupied, so the functional site signal is squeezed into the residual variance within each cluster.

**Panel 2 — Co-evolving position 4:** The dominant structure. The VAE must encode which of the 7 amino acids is at the (4, 14) pair — getting it wrong costs double reconstruction loss (both positions wrong). This appears as 7 clearly separated bands along one axis.

**Panel 3 — Co-evolving position 9:** The same logic for the (9, 19) pair, but encoded along the *orthogonal* axis. Compare panels 2 and 3: the 7-way stripes are rotated ~90°. The VAE has spontaneously disentangled two independent constraints into two independent latent dimensions.

**Panel 4 — Free position (pos 0):** No evolutionary constraint — all amino acids equally likely. Should look like uniform noise with no clustering.

**Key insight:** The clustering strength reflects *information content × coupling*. Co-evolving pairs (7 states, coupled) dominate; the functional site (2 states, uncoupled) leaves only a faint sub-structure. To see it cleanly you'd need a third latent dimension — try Exercise 2 (`latent_dim = 3`) and see if the functional site split sharpens.

## 5. Variant effect scoring

Now for the key idea: using the ELBO to score mutations.

We'll create two sets of variants from a single wildtype sequence:

| Variant type | Where | Why it should score low/high |
|---|---|---|
| **Grammar-breaking** | Functional site (pos 7 or 22): change to AA 2-6 | Violates evolutionary constraint — low ELBO |
| **Grammar-breaking** | Co-evolving pair: change pos 4 but NOT pos 14 | Breaks co-evolution coupling — low ELBO |
| **Neutral** | Free position: any amino acid change | No constraint violated — ELBO close to wildtype |

We compute:
$$\Delta\text{ELBO} = \text{ELBO}(\text{mutant}) - \text{ELBO}(\text{wildtype})$$

Negative $\Delta$ELBO = the mutant is less probable under the evolutionary model = likely pathogenic.

In [ ]:
def score_variants(model, wildtype, variants, device, n_samples=50):
    """
    wildtype: (L,) integer array
    variants: list of (L,) integer arrays
    Returns: array of delta ELBO scores (mutant - wildtype)
    """
    model.eval()
    all_seqs = np.vstack([wildtype[None], np.array(variants)])  # (1+N_var, L)
    oh = torch.tensor(one_hot(all_seqs, V), device=device)
    idx = torch.tensor(all_seqs, dtype=torch.long, device=device)

    with torch.no_grad():
        elbos = model.elbo(oh, idx, n_samples=n_samples).cpu().numpy()

    elbo_wt = elbos[0]
    delta_elbos = elbos[1:] - elbo_wt
    return delta_elbos, elbo_wt


# Build wildtype: a valid sequence
wildtype = generate_sequence(rng)
print(f"Wildtype at functional site 7:  {wildtype[7]} (should be 0 or 1)")
print(f"Wildtype co-evolving pair (4,14): {wildtype[4]}, {wildtype[14]} (should match)")

# -------------------------------------------------------
# Grammar-breaking variants
# -------------------------------------------------------
gb_variants = []
gb_labels = []

# 1. Mutate functional site 7 to amino acids 2-6 (forbidden)
for aa in range(2, V):
    mut = wildtype.copy()
    mut[7] = aa
    gb_variants.append(mut)
    gb_labels.append(f'pos7→{aa} (func site)')

# 2. Break co-evolution pair (4,14): change pos 4 but not pos 14
for aa in range(V):
    if aa == wildtype[4]: continue
    mut = wildtype.copy()
    mut[4] = aa  # pos 4 changed, pos 14 stays — coupling broken
    gb_variants.append(mut)
    gb_labels.append(f'pos4→{aa},pos14unchanged (coev break)')

# -------------------------------------------------------
# Neutral variants (free positions: 0, 1, 2, 3)
# -------------------------------------------------------
neutral_variants = []
neutral_labels = []
for pos in [0, 1, 2, 3]:
    for aa in range(V):
        if aa == wildtype[pos]: continue
        mut = wildtype.copy()
        mut[pos] = aa
        neutral_variants.append(mut)
        neutral_labels.append(f'pos{pos}→{aa} (neutral)')

# Score all variants
delta_gb, elbo_wt = score_variants(model, wildtype, gb_variants, device)
delta_neutral, _ = score_variants(model, wildtype, neutral_variants, device)

print(f"\nWildtype ELBO: {elbo_wt:.2f}")
print(f"Grammar-breaking ΔELBOs: mean={delta_gb.mean():.2f}, min={delta_gb.min():.2f}")
print(f"Neutral ΔELBOs:          mean={delta_neutral.mean():.2f}, min={delta_neutral.min():.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Panel 1: ΔELBO distributions
ax = axes[0]
bins = np.linspace(
    min(delta_gb.min(), delta_neutral.min()) - 0.5,
    max(delta_gb.max(), delta_neutral.max()) + 0.5,
    30
)
ax.hist(delta_neutral, bins=bins, alpha=0.7, color='steelblue',
        label=f'Neutral (n={len(neutral_variants)})')
ax.hist(delta_gb, bins=bins, alpha=0.7, color='firebrick',
        label=f'Grammar-breaking (n={len(gb_variants)})')
ax.axvline(0, color='k', lw=1.5, linestyle='--', label='Wildtype (ΔELBO=0)')
ax.set_xlabel('ΔELBO (mutant − wildtype)')
ax.set_ylabel('Count')
ax.set_title('Variant effect scores by type')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Panel 2: individual grammar-breaking scores
ax = axes[1]
colors_gb = ['C3' if 'func' in l else 'C1' for l in gb_labels]
y_pos = np.arange(len(delta_gb))
ax.barh(y_pos, delta_gb, color=colors_gb, alpha=0.8)
ax.axvline(0, color='k', lw=1)
ax.set_yticks(y_pos)
ax.set_yticklabels(gb_labels, fontsize=7)
ax.set_xlabel('ΔELBO')
ax.set_title('Grammar-breaking variants (individual)')

from matplotlib.patches import Patch
legend_patches = [
    Patch(color='C3', alpha=0.8, label='Functional site violation'),
    Patch(color='C1', alpha=0.8, label='Co-evolution coupling broken'),
]
ax.legend(handles=legend_patches, fontsize=8, loc='lower right')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.suptitle('VAE variant scoring: does ΔELBO separate neutral from damaging?', fontsize=12, y=1.02)
plt.show()

## 6. Mutation effect map

Let's compute the ELBO score for **every possible single amino acid substitution** across all 30 positions.  
This produces a **mutation effect map**: a (L × V) matrix of $\Delta$ELBO values.

In real EVE, this is done for all 19 possible substitutions at each of ~300 positions of a full-length human protein — producing thousands of variant scores in one pass.

In [ ]:
# Build all single-position substitutions
all_variants = []
pos_map = []
aa_map = []

for pos in range(L):
    for aa in range(V):
        if aa == wildtype[pos]: continue
        mut = wildtype.copy()
        mut[pos] = aa
        all_variants.append(mut)
        pos_map.append(pos)
        aa_map.append(aa)

delta_all, _ = score_variants(model, wildtype, all_variants, device, n_samples=30)

# Reshape into (L, V) grid (wildtype position = 0 by convention)
effect_map = np.full((L, V), np.nan)
for i, (pos, aa) in enumerate(zip(pos_map, aa_map)):
    effect_map[pos, aa] = delta_all[i]

# Mark wildtype positions with 0
for pos in range(L):
    effect_map[pos, wildtype[pos]] = 0.0

# Plot
fig, ax = plt.subplots(figsize=(14, 4))
vmax = np.nanpercentile(np.abs(delta_all), 95)
im = ax.imshow(effect_map.T, aspect='auto', cmap='RdBu', vmin=-vmax, vmax=vmax, origin='lower')
plt.colorbar(im, ax=ax, label='ΔELBO (blue=neutral, red=damaging)')

# Mark wildtype
for pos in range(L):
    ax.scatter(pos, wildtype[pos], color='black', s=12, zorder=5)

# Mark special positions
for i, j in coevolving_pairs:
    ax.axvline(i, color='red', lw=1.5, alpha=0.6, ls='--')
    ax.axvline(j, color='red', lw=1.5, alpha=0.6, ls='--')
for pos in functional_sites:
    ax.axvline(pos, color='orange', lw=1.5, alpha=0.6, ls=':')

ax.set_xlabel('Sequence position')
ax.set_ylabel('Amino acid')
ax.set_title('Mutation effect map: ΔELBO for every single substitution\n'
             '(■ = wildtype | red dashed = co-evolving | orange dotted = functional site)')
ax.set_yticks(range(V))
ax.set_yticklabels([f'AA {i}' for i in range(V)], fontsize=8)
plt.tight_layout()
plt.show()

## Key takeaways

1. **A VAE trained on an MSA learns evolutionary grammar** — constraints on amino acids at individual positions and between co-evolving positions emerge from the data, without explicit programming.

2. **The ELBO is a natural variant score**: variants that violate the grammar have lower ELBO under the trained model.

3. **The latent space organises sequences by their grammar class** — even with only 2 latent dimensions, the VAE separates sequences with different amino acids at functional and co-evolving positions.

4. **The mutation effect map** shows where the model has learned that mutations are tolerated (blue, $\Delta$ELBO ≈ 0) vs. damaging (red, $\Delta$ELBO < 0). The pattern is interpretable: functional sites and co-evolving positions are the most constrained.

5. **EVE scales this to real proteins**: in the published version, the MSA contains millions of homologous sequences from across the tree of life, and the VAE is larger — but the principle is identical.

## Exercises

### Exercise 1 — Add a third co-evolving pair
Add `(2, 25)` to `coevolving_pairs`. Retrain the VAE.  
**Q:** Does the new pair show up as a constrained region in the mutation effect map?  
**Q:** With 3 co-evolving pairs but only 2 latent dimensions, what do you expect to see in the latent space?

### Exercise 2 — KL vs reconstruction *(Advanced)*
Modify `score_variants` to return the reconstruction and KL terms separately.  
**Q:** For grammar-breaking variants, which term drives the low ELBO?  
**Q:** What does this tell you about where the evolutionary memory is stored in the model?

---
### Answer 1 — Third co-evolving pair

In [ ]:
# ── Answer 1: add a third co-evolving pair (2, 25) ──────────────────────────

# Step 1: rebuild the MSA with the new grammar rule
coevolving_pairs_ex1 = [(4, 14), (9, 19), (2, 25)]   # added (2, 25)

def generate_sequence_ex1(rng):
    seq = rng.integers(0, V, size=L)
    for i, j in coevolving_pairs_ex1:
        seq[j] = seq[i]
    for pos in functional_sites:
        seq[pos] = rng.choice(allowed_at_functional[pos])
    return seq

rng_ex1 = np.random.default_rng(42)
msa_ex1 = np.array([generate_sequence_ex1(rng_ex1) for _ in range(N_train)])
print("Match rates:")
for i, j in coevolving_pairs_ex1:
    print(f"  ({i},{j}): {(msa_ex1[:,i]==msa_ex1[:,j]).mean():.2f}")

# Step 2: re-encode and retrain
X_oh_ex1  = torch.tensor(one_hot(msa_ex1, V))
X_idx_ex1 = torch.tensor(msa_ex1, dtype=torch.long)
loader_ex1 = DataLoader(TensorDataset(X_oh_ex1, X_idx_ex1), batch_size=256, shuffle=True)

model_ex1     = ProteinVAE(seq_len=L, vocab_size=V, latent_dim=2, hidden_dim=256).to(device)
optimizer_ex1 = torch.optim.Adam(model_ex1.parameters(), lr=1e-3)

for epoch in range(1, 81):
    model_ex1.train()
    for x_oh, x_idx in loader_ex1:
        x_oh, x_idx = x_oh.to(device), x_idx.to(device)
        logits, mu, log_var, _ = model_ex1(x_oh)
        loss = compute_elbo_loss(logits, x_idx, mu, log_var)
        optimizer_ex1.zero_grad(); loss.backward(); optimizer_ex1.step()
    if epoch % 20 == 0:
        print(f"Epoch {epoch} | loss: {loss.item():.3f}")

# Step 3: visualise — with 3 pairs but only 2 latent dims, the model can't fully disentangle all three.
# Two pairs will dominate (whichever carry the most information / reconstruction pressure).
# The third will appear as a weaker sub-structure within the dominant clusters.
model_ex1.eval()
with torch.no_grad():
    mu_ex1 = model_ex1.encode(X_oh_ex1.to(device))[0].cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (pos, title) in zip(axes, [
    (4,  'Co-evolving pos 4 (4,14)'),
    (9,  'Co-evolving pos 9 (9,19)'),
    (2,  'Co-evolving pos 2 (2,25) — NEW'),
]):
    idx = rng.choice(N_train, 2000, replace=False)
    sc = ax.scatter(mu_ex1[idx,0], mu_ex1[idx,1], c=msa_ex1[idx,pos],
                    cmap='tab10', vmin=0, vmax=V-1, alpha=0.5, s=8)
    plt.colorbar(sc, ax=ax)
    ax.set_title(title, fontsize=9); ax.set_xlabel('z₁'); ax.set_ylabel('z₂')
plt.suptitle('Exercise 1: 3 co-evolving pairs, 2 latent dims\n'
             'Two pairs disentangle cleanly; the third is compressed into residual structure', fontsize=10)
plt.tight_layout(); plt.show()

---
### Answer 2 — KL vs reconstruction decomposition *(Advanced)*

In [ ]:
# ── Answer 4: decompose ΔELBO into reconstruction and KL terms ───────────────

def score_variants_decomposed(model, wildtype, variants, device, n_samples=50):
    """Returns ΔELBO, Δreconstruction, and ΔKL separately for each variant."""
    model.eval()
    all_seqs = np.vstack([wildtype[None], np.array(variants)])   # wildtype first, then variants
    oh  = torch.tensor(one_hot(all_seqs, V), device=device)
    idx = torch.tensor(all_seqs, dtype=torch.long, device=device)

    with torch.no_grad():
        mu, log_var = model.encode(oh)

        # Reconstruction term: E[log p(x|z)] estimated by Monte Carlo
        recon_sum = torch.zeros(oh.shape[0], device=device)
        for _ in range(n_samples):
            z      = model.reparameterise(mu, log_var)
            logits = model.decode(z)
            log_px_z = -F.cross_entropy(
                logits.view(-1, V), idx.view(-1), reduction='none'
            ).view(oh.shape[0], L).sum(dim=1)
            recon_sum += log_px_z
        recon = (recon_sum / n_samples).cpu().numpy()   # (N_variants+1,)

        # KL term: KL(q(z|x) || p(z))
        kl = (-0.5 * (1 + log_var - mu.pow(2) - log_var.exp()).sum(dim=1)).cpu().numpy()

    elbo = recon - kl

    # Compute deltas relative to wildtype (index 0)
    delta_elbo  = elbo[1:]  - elbo[0]
    delta_recon = recon[1:] - recon[0]
    delta_kl    = kl[1:]    - kl[0]   # positive ΔKL = mutant pushes further from prior

    return delta_elbo, delta_recon, delta_kl


# Score grammar-breaking and neutral variants with decomposed terms
d_elbo_gb, d_recon_gb, d_kl_gb          = score_variants_decomposed(model, wildtype, gb_variants,      device)
d_elbo_neu, d_recon_neu, d_kl_neu       = score_variants_decomposed(model, wildtype, neutral_variants, device)

# Plot the decomposition
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, d_elbo, d_recon, d_kl, label, color in [
    (axes[0], d_elbo_gb,  d_recon_gb,  d_kl_gb,  'Grammar-breaking', 'firebrick'),
    (axes[1], d_elbo_neu, d_recon_neu, d_kl_neu, 'Neutral',           'steelblue'),
]:
    x = np.arange(len(d_elbo))
    ax.bar(x,        d_recon, label='Δreconstruction', color='C0', alpha=0.8)
    ax.bar(x,       -d_kl,    label='−ΔKL',            color='C1', alpha=0.8, bottom=d_recon)
    ax.axhline(0, color='k', lw=1)
    ax.set_title(f'{label} variants\nΔELBO decomposed into reconstruction and KL', fontsize=9)
    ax.set_xlabel('Variant index'); ax.set_ylabel('ΔELBO contribution')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.suptitle('Where does the low ELBO come from? Reconstruction failure or posterior shift?',
             fontsize=11, y=1.02)
plt.show()

print("\nInterpretation:")
print("  Grammar-breaking variants: low ΔELBO is driven mainly by the RECONSTRUCTION term.")
print("  The decoder can't reconstruct the sequence well because the latent code z")
print("  doesn't correspond to any region the model learned to decode.")
print("  → The evolutionary memory lives in the DECODER (p(x|z)), not the encoder.")
print()
print("  Neutral variants: both terms near 0 — the model encodes and decodes them without surprise.")